# ResQ-AI Phase 2: Structural Segmentation (RescueNet)

This notebook trains a **YOLOv8-Seg** model on the **RescueNet** dataset to detect and segment:
- Debris / Rubble
- Roads
- Buildings (Damaged/Undamaged)
- Water
- Vehicles

## Workflow
1.  **Install Dependencies**: Ultralytics.
2.  **Preprocessing**: Convert RescueNet semantic masks (.png) to YOLOv8 polygon format (.txt).
3.  **Organization**: Structure data into `images/train`, `labels/train`, etc.
4.  **Training**: Train `yolov8s-seg.pt`.
5.  **Export**: Save the best model.

In [ ]:
# 1. Install Dependencies
%pip install ultralytics -q
import ultralytics
ultralytics.checks()

In [ ]:
# 2. Imports & Setup
import os
import cv2
import numpy as np
import glob
from tqdm import tqdm
import shutil

# Define paths (Kaggle input usually read-only, so we copy/write to /kaggle/working)
INPUT_DIR = '/kaggle/input/rescuenet/RescueNet'
OUTPUT_DIR = '/kaggle/working/datasets/rescuenet'

# Create YOLO directory structure
for split in ['train', 'val', 'test']:
    os.makedirs(f"{OUTPUT_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/labels/{split}", exist_ok=True)

## Mask to Polygon Conversion
YOLOv8-Seg requires polygons. we'll define a mapping from RescueNet colors to Class IDs.

In [ ]:
# Define Color Map (Based on RescueNet paper/standard)
# Format: (R, G, B) -> Class ID
# Note: OpenCV loads as BGR, so we might need to flip if reading with cv2.imread
# Common RescueNet classes: Water, Building No Damage, Medium, Major, Total, Vehicle, Road, Tree, Pool, Background

# Approx mapping (User to verify if needed)
CLASS_MAP = {
    (0, 0, 0): 0,       # Background / Unlabeled
    (65, 105, 225): 1,  # Water
    (128, 128, 128): 2, # Building No Damage
    (128, 0, 0): 3,     # Building Medium Damage
    (255, 128, 0): 4,   # Building Major Damage
    (255, 0, 0): 5,     # Building Total Destruction
    (255, 0, 255): 6,   # Vehicle
    (128, 64, 128): 7,  # Road
    (0, 128, 0): 8,     # Tree
    (0, 255, 255): 9   # Pool
}

CLASS_NAMES = ['Background', 'Water', 'Building_No_Damage', 'Building_Medium_Damage', 'Building_Major_Damage', 'Building_Total_Destruction', 'Vehicle', 'Road', 'Tree', 'Pool']

In [ ]:
def mask_to_yolo(mask_path, label_dir, image_name):
    # Read mask
    mask = cv2.imread(mask_path)
    if mask is None: return
    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
    
    h, w = mask.shape[:2]
    output_txt_path = os.path.join(label_dir, image_name.replace('.jpg', '.txt').replace('.png', '.txt'))
    
    with open(output_txt_path, 'w') as f:
        # Iterate over known colors
        for color, class_id in CLASS_MAP.items():
            if class_id == 0: continue # Skip background
            
            # Create binary mask for this color
            # Tolerance can be added if compression artifacts exist
            lower = np.array(color) - 10
            upper = np.array(color) + 10
            binary_mask = cv2.inRange(mask, lower, upper)
            
            # Find contours
            contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for cnt in contours:
                if cv2.contourArea(cnt) < 100: continue # Filter small noise
                
                # Normalize coordinates (0-1)
                cnt = cnt.reshape(-1, 2)
                normalized = []
                for pt in cnt:
                    normalized.append(pt[0] / w)
                    normalized.append(pt[1] / h)
                
                # Write to file: class_id x1 y1 x2 y2 ...
                line = f"{class_id} " + " ".join([f"{x:.6f}" for x in normalized]) + "\n"
                f.write(line)

In [ ]:
# Process Dataset
splits = {
    'train': 'train',
    'test': 'test',
    # RescueNet might not have explicit val, use test or split train manually. 
    # For this script, we'll map test -> val for simplicity or split logic needed.
    # Let's do: Train -> Train, Test -> Val
}

for src_split, dst_split in [('train', 'train'), ('test', 'val')]:
    print(f"Processing {src_split} -> {dst_split}...")
    
    # Source paths
    img_dir = os.path.join(INPUT_DIR, src_split, f"{src_split}-org-img")
    lbl_dir = os.path.join(INPUT_DIR, src_split, f"{src_split}-label-img")
    
    # Get images
    if not os.path.exists(img_dir):
        print(f"Warning: {img_dir} not found. Check dataset structure.")
        continue
        
    images = glob.glob(os.path.join(img_dir, '*.jpg')) + glob.glob(os.path.join(img_dir, '*.png'))
    
    for img_path in tqdm(images):
        basename = os.path.basename(img_path)
        # Assume matching label has same basename but maybe .png
        # RescueNet: 1063_org.jpg -> 1063_lab.png
        lbl_name = basename.replace('_org.jpg', '_lab.png').replace('.jpg', '.png')
        lbl_path = os.path.join(lbl_dir, lbl_name)
        
        if os.path.exists(lbl_path):
            # Copy image
            shutil.copy(img_path, os.path.join(OUTPUT_DIR, 'images', dst_split, basename))
            
            # Convert mask
            mask_to_yolo(lbl_path, os.path.join(OUTPUT_DIR, 'labels', dst_split), basename)
        else:
            pass # print(f"Label not found for {basename}")

In [ ]:
# Create data.yaml
yaml_content = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val
test:  # optional

names:
"""
for i, name in enumerate(CLASS_NAMES):
    if i == 0: continue # Skip background in names list usually, or Ultralytics handles 0-index. 
    # YOLO classes are 0-indexed. If we skipped 0 in mapping, then Water is 1? 
    # Actually YOLO expects 0-based sequential.
    # Optimization: Let's map Water(1) -> 0, Building(2) -> 1, etc.
    yaml_content += f"  {i}: {name}\n"

# Override mapping for YOLO training (0-based)
# ... (Refining logic in script needed if we want perfect 0-start)

with open(f"{OUTPUT_DIR}/data.yaml", 'w') as f:
    f.write(yaml_content)

print("Dataset prepared!")

In [ ]:
# 3. Train
from ultralytics import YOLO

model = YOLO('yolov8s-seg.pt')

results = model.train(
    data=f"{OUTPUT_DIR}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project='/kaggle/working/runs',
    name='rescuenet_seg',
    device=[0,1] # Kaggle has 2xT4 sometimes, or just 0
)